# 🚀 Pipeline Maestro: Data Science & AI - Tumipay
Este cuaderno es la puerta de entrada principal. Orquesta todo el flujo desde la carga de datos crudos hasta el entrenamiento del modelo LightGBM, la persistencia en base de datos para consumo en Power BI, y la interacción con el Agente RAG.

In [27]:
# 0. Configuración Inicial y Paths
import sys
import os
import importlib
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine

# Agregar src al path dinámicamente
current_dir = Path(os.getcwd())
ROOT_DIR = current_dir if (current_dir / 'src').exists() else current_dir.parent

if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

# Forzar recarga de config para garantizar que los campos nuevos están disponibles
import src.config as _cfg_module
importlib.reload(_cfg_module)
from src.config import settings

from src.ingesta import DataIngestor
from src.procesamiento import DataCleaner
from src.consolidacion import DataConsolidator
from src.train_mora import MoraModelTrainer

%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 🏗️ 1. ETL: Ingesta, Limpieza y Consolidación (ABT)

In [13]:
# 1. ETL: Ingesta, Limpieza y Consolidación → ABT

# raw_dir es el parámetro correcto (no data_dir)
ingestor = DataIngestor(raw_dir=ROOT_DIR / "data" / "raw")
dfs = ingestor.load_all()

cleaner = DataCleaner()
df_clientes = cleaner.clean_clientes(dfs['clientes'])
df_creditos = cleaner.clean_creditos(dfs['creditos'])
df_pagos    = cleaner.clean_pagos(dfs['pagos'])

# cutoff_date es requerido en DataConsolidator
consolidator = DataConsolidator(cutoff_date=settings.CUTOFF_DATE)
abt = consolidator.build_analytical_base_table(df_clientes, df_creditos, df_pagos)

# Guardar checkpoint para el pipeline de entrenamiento
settings.DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
abt.to_parquet(settings.DATA_PROCESSED_DIR / "abt.parquet", index=False)

print(f"✅ ABT lista: {abt.shape[0]} registros, {abt.shape[1]} variables.")
print(f"💾 Checkpoint guardado en: {settings.DATA_PROCESSED_DIR / 'abt.parquet'}")
display(abt.head(3))


2026-05-20 11:41:04,065 - src.ingesta - INFO - Cargando todos los archivos del dataset...
2026-05-20 11:41:04,066 - src.ingesta - INFO - Iniciando carga de archivo: /home/drodriguez/Documentos/proyectos/proyecto-data-scientist/data/raw/clientes.csv
2026-05-20 11:41:04,076 - src.ingesta - INFO - Columna 'fecha_registro' casteada a datetime64[ns].
2026-05-20 11:41:04,076 - src.ingesta - INFO - Archivo clientes.csv cargado exitosamente. Forma: (1400, 16)
2026-05-20 11:41:04,077 - src.ingesta - INFO - Iniciando carga de archivo: /home/drodriguez/Documentos/proyectos/proyecto-data-scientist/data/raw/creditos.csv
2026-05-20 11:41:04,085 - src.ingesta - INFO - Columna 'fecha_desembolso' casteada a datetime64[ns].
2026-05-20 11:41:04,086 - src.ingesta - INFO - Archivo creditos.csv cargado exitosamente. Forma: (1527, 13)
2026-05-20 11:41:04,087 - src.ingesta - INFO - Iniciando carga de archivo: /home/drodriguez/Documentos/proyectos/proyecto-data-scientist/data/raw/pagos.csv


2026-05-20 11:41:04,119 - src.ingesta - INFO - Columna 'fecha_vencimiento' casteada a datetime64[ns].
2026-05-20 11:41:04,126 - src.ingesta - INFO - Columna 'fecha_pago' casteada a datetime64[ns].
2026-05-20 11:41:04,127 - src.ingesta - INFO - Archivo pagos.csv cargado exitosamente. Forma: (10434, 11)
2026-05-20 11:41:04,128 - src.ingesta - INFO - Iniciando carga de archivo: /home/drodriguez/Documentos/proyectos/proyecto-data-scientist/data/raw/eventos_app.csv
2026-05-20 11:41:04,164 - src.ingesta - INFO - Columna 'fecha_evento' casteada a datetime64[ns].
2026-05-20 11:41:04,165 - src.ingesta - INFO - Archivo eventos_app.csv cargado exitosamente. Forma: (13324, 8)
2026-05-20 11:41:04,166 - src.procesamiento - INFO - Iniciando procesamiento y limpieza de tabla 'clientes'.
2026-05-20 11:41:04,169 - src.procesamiento - INFO - Imputados valores nulos en 'ingreso_mensual_estimado' utilizando la mediana.
2026-05-20 11:41:04,170 - src.procesamiento - INFO - Imputados valores nulos en 'score_e

✅ ABT lista: 1527 registros, 29 variables.
💾 Checkpoint guardado en: /home/drodriguez/Documentos/proyectos/proyecto-data-scientist/data/processed/abt.parquet


,credito_id,cliente_id,fecha_desembolso,producto_credito,monto_credito,plazo_meses,tasa_interes_mensual,valor_cuota_pactada,canal_originacion,score_interno_originacion,...,estrato,nivel_educativo,ocupacion,ingreso_mensual_estimado,canal_adquisicion,score_externo,tiene_producto_ahorro,numero_dependientes,dispositivo_principal,email_hash
0,CR000001,CL00001,2025-03-25,Crédito libre inversión,2000000,9,0.0301,257000,App,677,...,1,Técnico/Tecnólogo,Comerciante,1600000.0,Campaña digital,444.0,False,2,Android,hash_732413
1,CR000002,CL00002,2025-09-08,Avance de nómina,2150000,3,0.0261,754000,Web,859,...,3,Técnico/Tecnólogo,Pensionado,3310000.0,Referido,580.0,False,3,iOS,hash_130099
2,CR000003,CL00003,2024-06-28,Microcrédito,4350000,6,0.0398,829000,Aliado,805,...,2,Universitario,Independiente,2350000.0,Campaña digital,564.0,False,0,iOS,hash_166668


## 🧠 2. Entrenamiento y Scraping de Resultados LightGBM

In [14]:
# 2. Entrenamiento LightGBM con MoraModelTrainer
import joblib

settings.MODELS_DIR.mkdir(parents=True, exist_ok=True)

trainer = MoraModelTrainer(
    data_path=settings.DATA_PROCESSED_DIR / "abt.parquet",
    model_output_path=settings.MODELS_DIR / "modelo_mora.pkl",
)

# Partir dataset, entrenar y evaluar
X_train, X_test, y_train, y_test = trainer.train_test_split_custom(abt)
model = trainer.train_model(X_train, y_train)
trainer.evaluate_model(model, X_test, y_test)
trainer.export_model(model, list(X_train.columns))
print(f"\n✅ Modelo guardado en: {settings.MODELS_DIR / 'modelo_mora.pkl'}")

# Añadir probabilidades al dataset completo (mismo preprocesamiento que en entrenamiento)
X_full = abt[X_train.columns].copy()
for col in X_full.select_dtypes(include=['object', 'str']).columns:
    X_full[col] = X_full[col].astype('category')

abt['probabilidad_mora'] = model.predict_proba(X_full)[:, 1]
abt['prediccion_mora']   = model.predict(X_full)

print("\n🎯 Primeras predicciones:")
display(abt[['credito_id', 'es_moroso', 'probabilidad_mora', 'prediccion_mora']].head(5))


2026-05-20 11:41:18,435 - src.train_mora - INFO - Dataset particionado. Train=1221, Test=306.
2026-05-20 11:41:18,436 - src.train_mora - INFO - Iniciando entrenamiento del modelo LightGBM...


[LightGBM] [Info] Number of positive: 309, number of negative: 912
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001416 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1757
[LightGBM] [Info] Number of data points in the train set: 1221, number of used features: 24
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -i

2026-05-20 11:41:18,711 - src.train_mora - INFO - Entrenamiento finalizado.
2026-05-20 11:41:18,712 - src.train_mora - INFO - Evaluando el modelo...
2026-05-20 11:41:18,722 - src.train_mora - INFO - 
Matriz de Confusión:
[[220   8]
 [  0  78]]
2026-05-20 11:41:18,726 - src.train_mora - INFO - 
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.96      0.98       228
           1       0.91      1.00      0.95        78

    accuracy                           0.97       306
   macro avg       0.95      0.98      0.97       306
weighted avg       0.98      0.97      0.97       306

2026-05-20 11:41:18,728 - src.train_mora - INFO - ROC-AUC Score: 0.9844
2026-05-20 11:41:18,730 - src.train_mora - INFO - 
Top 5 Variables más importantes:
                 Feature  Importance
    tasa_interes_mensual         230
           monto_credito         188
           score_externo         179
ingreso_mensual_estimado         167
              

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

,credito_id,es_moroso,probabilidad_mora,prediccion_mora
0,CR000001,0,0.004307,0
1,CR000002,0,0.004937,0
2,CR000003,1,0.901645,1
3,CR000004,0,0.004993,0
4,CR000005,0,0.004377,0


## 💾 3. Almacenamiento en PostgreSQL (Directo para Power BI)
Bajo las mejores prácticas de Arquitectura BI, escribimos los resultados finales del dataset anotado (features + probabilidades de mora de LightGBM) en Postgres para ser consultadas por RAG y Dashboard.

In [15]:
# ⚠️ Asegúrate de tener el Docker Compose levantado antes de correr esto.
try:
    engine = create_engine(settings.DATABASE_URL)
    # Mandamos los resultados a la base de datos
    abt.to_sql('predicciones_riesgo', engine, if_exists='replace', index=False)
    print("✅ Resultados del modelo LightGBM inyectados con éxito en la tabla 'predicciones_riesgo' de PostgreSQL.")
    print("📊 Power BI ahora se puede conectar directamente apuntando a pg_database:5432")
except Exception as e:
    print(f"⚠️ Error conectando a BD. Valida tener expuesto el puerto 5432:\n{e}")

✅ Resultados del modelo LightGBM inyectados con éxito en la tabla 'predicciones_riesgo' de PostgreSQL.
📊 Power BI ahora se puede conectar directamente apuntando a pg_database:5432


## 🤖 4. Consulta al Asesor Financiero (RAG Text-to-SQL + LLM)

Consultamos los datos transaccionales, de demografía y riesgo usando la API de Minimax.

El knowledge base del RAG incluye:
- **1.527 fichas de perfil** (una por crédito) con datos del cliente, condiciones del crédito y predicción de mora
- **Resumen estadístico del portafolio** con KPIs globales, desglose por producto y canal
- **4 documentos de política** con reglas de riesgo, proceso de cobro y glosario financiero

> Para repoblar el vector store con datos frescos ejecuta la celda de abajo o corre:  
> `python scripts/populate_rag.py`

In [34]:
import logging, os, sys, warnings, subprocess
from IPython.utils import io as _ipio

# Silenciar librerías verbosas antes de cualquier importación
os.environ["TQDM_DISABLE"] = "1"
warnings.filterwarnings("ignore")
for _lib in ["httpx", "httpcore", "sentence_transformers", "transformers",
             "huggingface_hub", "langchain", "langchain_core",
             "langchain_community", "src.rag_agent", "openai"]:
    logging.getLogger(_lib).setLevel(logging.ERROR)

# ── 4a. Poblar el RAG Knowledge Base ─────────────────────────────────────────
print("📥 Poblando RAG knowledge base...", end=" ", flush=True)
with _ipio.capture_output():
    result = subprocess.run(
        [sys.executable, "scripts/populate_rag.py"],
        capture_output=True, text=True,
        env={**os.environ, "TQDM_DISABLE": "1"},
    )
if result.returncode != 0:
    print(f"⚠️ Error (código {result.returncode})\n{result.stderr[-400:]}")
else:
    for line in reversed((result.stdout + result.stderr).splitlines()):
        if "chunks" in line or "✅" in line:
            print(line.split(" - ")[-1])
            break

# ── 4b. Consulta al Asesor Financiero RAG ────────────────────────────────────
from src.rag_agent import RAGAgentPipeline

print("🔧 Iniciando agente RAG...", end=" ", flush=True)
with _ipio.capture_output():
    agent = RAGAgentPipeline(
        db_connection=settings.DATABASE_URL,
        nvidia_api_key=settings.NVIDIA_API_KEY,
    )
print("✅\n")

preguntas = [
    "¿Cuál es la tasa de mora actual del portafolio y cuántos clientes están en riesgo crítico?",
    "¿Qué pasos sigue TumiPay para gestionar a un cliente con mora mayor a 30 días?",
]

for pregunta in preguntas:
    print(f"🧑 {pregunta}")
    print("🤖 ", end="", flush=True)
    for chunk in agent.stream_query(pregunta):
        print(chunk, end="", flush=True)
    print("\n" + "─" * 60 + "\n")


📥 Poblando RAG knowledge base... ✅  RAG Knowledge Base poblado con 1538 chunks.
🔧 Iniciando agente RAG... ✅

🧑 ¿Cuál es la tasa de mora actual del portafolio y cuántos clientes están en riesgo crítico?
🤖 

# Información Solicitada — Fecha de Corte: 2026-04-30

---

## 📊 Tasa de Mora del Portafolio

| Indicador | Valor |
|---|---|
| Clientes morosos | **387** |
| Total créditos en portafolio | 1,527 |
| **Tasa de mora** | **25.3%** |
| Monto en riesgo (morosos) | $991,650,000 |

La tasa de mora representa clientes con al menos **una cuota con DPD > 30 días**.

---

## ⚠️ Clientes en Riesgo Crítico

**No dispongo de esa información en los expedientes actuales.**

El contexto disponible incluye la definición operativa del nivel **Crítico (PD ≥ 0.80)**, que activa intervención inmediata y cobro activo, pero **no proporciona el conteo desagregado** de clientes por nivel de riesgo.

El modelo predictivo (LightGBM) clasifica la probabilidad de mora (PD) en cuatro niveles:

| Nivel | Rango de 